# Redo 16 Diverse Plates — Plate 4 — 3×3 (slice,hatch) SEM labeling

Labels are stored in an **append-only** CSV log under `data/labels/redo_16_diverse_plates/` and can be resumed safely.
This notebook never edits the plate `tracking.csv`.


In [ ]:
%matplotlib inline
import csv
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display, clear_output

# --- Locate repo root (search upward for data/redo_16_diverse_plates) ---
NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_DIR
for _cand in [NOTEBOOK_DIR] + list(NOTEBOOK_DIR.parents):
    if (_cand / 'data' / 'redo_16_diverse_plates').exists():
        REPO_ROOT = _cand
        break

PLATE_ID = 4
PLATE_DIR = REPO_ROOT / 'data' / 'redo_16_diverse_plates' / f'redo_16_diverse_plate_{PLATE_ID}'
MASTER_CSV_PATH = PLATE_DIR / 'tracking.csv'   # read-only
RENDERS_DIR = PLATE_DIR / 'renders'

LABEL_ROOT = REPO_ROOT / 'data' / 'labels' / 'redo_16_diverse_plates'
DERIVED_DIR = LABEL_ROOT / 'derived'
LABEL_ROOT.mkdir(parents=True, exist_ok=True)
DERIVED_DIR.mkdir(parents=True, exist_ok=True)

LABEL_LOG_PATH = LABEL_ROOT / f'plate_{PLATE_ID}_labels_log.csv'
ML_DATASET_PATH = DERIVED_DIR / f'plate_{PLATE_ID}_ml_dataset_derived.csv'

# Make repo importable (so prompt2cad.* imports work)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import importlib
import prompt2cad.labeling.sem_validation as sem_validation
importlib.reload(sem_validation)
from prompt2cad.labeling.sem_validation import FabricationSweepLabelStore

# 3x3 sweep definition (slice_um × hatch_um)
SLICE_VALUES = [0.2, 0.3, 0.4]
HATCH_VALUES = [0.2, 0.3, 0.4]
ROUND_MAP = {}
_rid = 1
for s in SLICE_VALUES:
    for h in HATCH_VALUES:
        ROUND_MAP[_rid] = (s, h)
        _rid += 1
ROUND_IDS = list(range(1, 10))


In [ ]:
# ==========================================
# STATE MANAGEMENT
# ==========================================
round_metadata = {rid: {"slice_um": ROUND_MAP[rid][0], "hatch_um": ROUND_MAP[rid][1]} for rid in ROUND_IDS}

store = FabricationSweepLabelStore(
    master_csv_path=MASTER_CSV_PATH,
    label_log_path=LABEL_LOG_PATH,
    ml_dataset_path=ML_DATASET_PATH,
    round_ids=ROUND_IDS,
    round_metadata=round_metadata,
)
model_ids = store.model_ids
state = store.load_latest_state()

def save_progress():
    # Durable labels are in LABEL_LOG_PATH; this export is derived.
    store.export_ml_dataset()

summary = store.summary()
print(f'Plate dir:            {PLATE_DIR}')
print(f'Tracking CSV (RO):    {summary["master_csv_path"]}')
print(f'Labels log (append):  {summary["label_log_path"]}')
print(f'Derived ML dataset:   {summary["ml_dataset_path"]}')
print(f'Models: {summary["model_count"]} | labeled: {summary["labeled_total"]} | by round: {summary["labeled_by_round"]}')
print('\nRound map (print_round -> (slice_um, hatch_um)):')
for rid in ROUND_IDS:
    s, h = ROUND_MAP[rid]
    print(f'  {rid}: slice={s} hatch={h}')

# Ensure derived CSV exists (header-only if no labels yet)
save_progress()


In [ ]:
# ==========================================
# RENDER DISPLAY + LABELING ENGINE
# ==========================================

def _show_images(paths):
    paths = [Path(p) for p in (paths or [])]
    if not paths:
        return
    fig, axes = plt.subplots(1, len(paths), figsize=(18, 5))
    if len(paths) == 1:
        axes = [axes]
    for ax, p in zip(axes, paths):
        ax.imshow(mpimg.imread(str(p)))
        ax.set_title(p.name, fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    display(fig)
    plt.close(fig)

def show_renders_for_model(model_id):
    mid = str(model_id).strip()
    if not RENDERS_DIR.exists():
        print(f'[WARN] Renders dir missing: {RENDERS_DIR}')
        return False

    picks = []
    for view in ['top', 'iso', 'front', 'side']:
        matches = sorted(RENDERS_DIR.glob(f'{mid}_*_{view}.png'))
        if matches:
            picks.append(matches[0])

    if not picks:
        any_matches = sorted(RENDERS_DIR.glob(f'{mid}_*.png'))
        picks = any_matches[:4]

    if picks:
        _show_images(picks)
        return True
    return False

def _tail_recent_labels(n=3):
    path = Path(LABEL_LOG_PATH)
    if not path.exists():
        return []
    try:
        with path.open('r', encoding='utf-8', newline='') as handle:
            rows = list(csv.DictReader(handle))
        return [str(r.get('model_id') or '').strip() for r in rows[-max(0, int(n)):] if str(r.get('model_id') or '').strip()]
    except Exception:
        return []

def run_labeling_session(print_round):
    print_round = int(print_round)
    if print_round not in ROUND_MAP:
        raise ValueError(f'Unknown print_round {print_round}')

    slice_um, hatch_um = ROUND_MAP[print_round]
    title = f'Round {print_round}: slice={slice_um} hatch={hatch_um}'

    global state
    state = store.load_latest_state()

    unlabeled = [mid for mid in model_ids if mid not in state[print_round]]
    print(f'Starting {title} | Remaining: {len(unlabeled)}')

    recent = _tail_recent_labels(n=3)
    if recent:
        print('\nRecent labeled geometries:')
        for mid in recent:
            print(f'  - {mid}')
            show_renders_for_model(mid)

    for i, mid in enumerate(unlabeled, start=1):
        clear_output(wait=True)
        print('=' * 60)
        print(f'[{title}] MODEL [{i}/{len(unlabeled)}]: {mid}')
        print('=' * 60)

        if not show_renders_for_model(mid):
            print(f'[WARN] No renders found for {mid} in {RENDERS_DIR}')
            raw = input('Skip [s] or Quit [q]? ').strip().lower()
            if raw[:1] == 'q':
                print('Quitting & Saving...')
                break
            continue

        while True:
            raw = input("Evaluate [p/f/s/q] (optional note after ':'): ").strip()
            if not raw:
                continue
            cmd = raw.lower().strip()[:1]
            note = raw[1:].lstrip(' :') if len(raw) > 1 else ''
            if cmd in ['p', 'f', 's', 'q']:
                action = cmd
                notes = note
                break

        if action == 'q':
            print('Quitting & Saving...')
            break
        if action == 's':
            continue

        label_val = 1 if action == 'p' else 0
        state[print_round][mid] = {'label': label_val, 'notes': notes}
        store.append_label(model_id=mid, print_round=print_round, validation_label=label_val, notes=notes)
        save_progress()

    save_progress()
    print(f'\n{title} session ended! Progress saved.')


## Run labeling sessions (3×3 sweep)
Run these cells in order. Stop anytime and resume later.


In [ ]:
# slice=0.2 hatch=0.2
run_labeling_session(1)


In [ ]:
# slice=0.2 hatch=0.3
run_labeling_session(2)


In [ ]:
# slice=0.2 hatch=0.4
run_labeling_session(3)


In [ ]:
# slice=0.3 hatch=0.2
run_labeling_session(4)


In [ ]:
# slice=0.3 hatch=0.3
run_labeling_session(5)


In [ ]:
# slice=0.3 hatch=0.4
run_labeling_session(6)


In [ ]:
# slice=0.4 hatch=0.2
run_labeling_session(7)


In [ ]:
# slice=0.4 hatch=0.3
run_labeling_session(8)


In [ ]:
# slice=0.4 hatch=0.4
run_labeling_session(9)
